In [ ]:
# ============================================================================
# 1. SETUP & CONFIGURATION
# ============================================================================
import os
import time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pencilbox as pb

# ---- Reporting period --------------------------------------------------------
start_date = '2026-04-01'
end_date   = '2026-04-30'

# ---- Configuration ----
OUTPUT_DIR = os.path.expanduser('~/data_extracts')
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNKSIZE = 100_000  # Number of rows to pull per batch (adjust based on memory constraints)

# ---- Connection ----
print("Establishing connection to Trino...")
# Uses the standard Pencilbox Trino connection from your environment
conn = pb.get_connection('[Warehouse] Trino')
print("Connection established successfully.")

In [ ]:
# ============================================================================
# 2. EXPORT ENGINE
# ============================================================================
def execute_and_export(query: str, filename: str, chunksize: int = CHUNKSIZE):
    """
    Executes a SQL query in chunks, displays a sample, and writes to a Parquet file.
    """
    output_path = os.path.join(OUTPUT_DIR, filename)
    print(f"Starting execution. Output will be saved to: {output_path}")
    
    start_time = time.time()
    writer = None
    total_rows = 0
    
    try:
        # Fetch data in chunks to prevent memory overload
        for i, chunk in enumerate(pd.read_sql_query(query, conn, chunksize=chunksize)):
            if len(chunk) == 0:
                continue
                
            # If it's the first chunk, display a sample and initialize the ParquetWriter
            if i == 0:
                print("\n--- Sample Data (First 5 Rows) ---")
                display(chunk.head())
                print("----------------------------------\n")
                
                # Convert first chunk to Arrow Table to infer schema
                table = pa.Table.from_pandas(chunk, preserve_index=False)
                writer = pq.ParquetWriter(output_path, table.schema)
            else:
                # Convert subsequent chunks
                table = pa.Table.from_pandas(chunk, preserve_index=False)
            
            # Write chunk to file
            writer.write_table(table)
            total_rows += len(chunk)
            print(f"Processed {total_rows:,} rows...")
            
    except Exception as e:
        print(f"An error occurred: {e}")
        
    finally:
        # Ensure the writer is properly closed even if an error occurs
        if writer:
            writer.close()
            
    elapsed_time = time.time() - start_time
    print(f"\nDone! Successfully exported {total_rows:,} rows in {elapsed_time:.1f} seconds.")

In [ ]:
# ============================================================================
# 3. RUN YOUR QUERY
# ============================================================================

# Define your custom query here
MY_QUERY = """
 WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names

    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),

raw as (
SELECT
    c.id  AS contract_id,
    c.state   AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,
    qa.id  AS question_answer_id,
    qa.question_id,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    json_parse(qa.answer) as answer

FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0 
      -- [DEBUG POINT 1]: Section Name Match
      -- Sample Failure: If the system saves this as 'satellite location fees' (lowercase) 
      -- or 'Satellite Location Fees ' (trailing space), this join will fail.
      -- Fix: AND LOWER(TRIM(s.name)) = 'satellite location fees'
      AND s.name = 'Satellite Location Fees'
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
-- [DEBUG POINT 2]: Contract State
-- Sample Failure: If the state is 'EXPIRED' or 'TERMINATED', they will pass this filter. 
-- Ensure you don't accidentally pull fees for contracts that are no longer active.
and c.state not in ('ARCHIVED','REJECTED')
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST
),

format_ as (
select contract_id,
-- [DEBUG POINT 3]: JSON Key Casing
-- Sample Failure: JSON is case-sensitive. If the raw JSON looks like {"manufacturer": {"id": 123}}, 
-- '$.Manufacturer.id' (capital M) will return NULL. Verify the raw JSON exact casing.
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,

contract_state,
created_at,
question_answer_id,
question_id,
json_extract_scalar(x,'$.value') as answer_values,
x,
manufacturer_ids,
manufacturer_names
from raw
cross join unnest (
    CASE 
        -- [DEBUG POINT 4]: JSON Array Casing
        -- Sample Failure: Same as above. If the JSON key is '$.Values' (Capital V), 
        -- this unnest will fail silently and drop the row.
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
),

details as (
select contract_id,
json_extract_scalar(row, '$.product_type') AS product_type,
json_extract_scalar(row, '$.bucket') AS bucket,
-- [DEBUG POINT 5]: Casting NULLs to Double
-- Sample Failure: If the JSON is missing the 'item_mrp' key, json_extract_scalar returns NULL. 
-- Casting NULL to Double is fine, but math later on (NULL * 1) will fail silently.
CAST(json_extract_scalar(row, '$.item_mrp') AS double) AS item_mrp,
json_extract_scalar(row, '$.charge_type') AS charge_type,
CAST(json_extract_scalar(row, '$.value') AS double) AS fee_amt,
json_extract_scalar(row, '$.max_fee_threshold') AS max_fee_threshold,
json_extract_scalar(row, '$.max_fee_amt') AS max_fee_amt
from format_
cross join unnest(
     CAST(json_extract(x, '$.value.rows') AS array(json))
    ) AS t(row)
where question_id='satellite_fees_table_v1'
),

sat_fee as (
select f.contract_id,f.manufacturer_id,
manufacturer,
contract_state,
min(answer_values) filter(where question_id='satellite_fees_enable_v1') as satellite_fee,
product_type,
bucket,
item_mrp,
charge_type,
fee_amt,
max_fee_threshold,
max_fee_amt
from format_ f
left join details d on f.contract_id=d.contract_id
group by 1,2,3,4,6,7,8,9,10,11,12
),

sf as (
select * 
from sat_fee 
-- [DEBUG POINT 6]: Boolean Text Match
-- Sample Failure: If the system stores 'yes', 'YES', or 'Yes ' (with a space), this drops everything.
-- Fix: where LOWER(TRIM(satellite_fee)) = 'yes'
where satellite_fee='Yes'
),

--------------------------------------------------------------------------------------------------------------------------
sat_city as (
with a as (select fe_city_name as city_name,fe_facility_id,fe_facility_name,
be_facility_id,
be_facility_name,
avg(wh_lat) as wh_lat,
        avg(wh_lon) as wh_lon,
        avg(ds_lat) as ds_lat,
        avg(ds_lon) as ds_lon,
111 * (
ABS(CAST(avg(wh_lat) AS DOUBLE) - CAST(avg(ds_lat) AS DOUBLE)) +
ABS(CAST(avg(wh_lon) AS DOUBLE) - CAST(avg(ds_lon)  AS DOUBLE))
) as dist
from supply_etls.inventory_metrics_tea_tagging tt
JOIN supply_etls.outlet_details AS od
            ON tt.be_facility_id = od.facility_id
            AND od.taggings = 'be'
            AND od.store_type = 'Packaged Goods'
JOIN supply_etls.wh_to_ds_distance_v4 d
ON d.ds_facility_id = tt.fe_facility_id
AND d.wh_facility_id = tt.be_facility_id
WHERE tt.flag = 'correct_tea'
AND tt.assortment_status_id in (1,3)
and tt.store_assortment_type = 'EXPRESS'
group by 1,2,3,4,5
),

city_level as (
select city_name,avg(wh_lat) as wh_lat,
        avg(wh_lon) as wh_lon,
        avg(ds_lat) as ds_lat,
        avg(ds_lon) as ds_lon,
        111 * (
ABS(CAST(avg(wh_lat) AS DOUBLE) - CAST(avg(ds_lat) AS DOUBLE)) +
ABS(CAST(avg(wh_lon) AS DOUBLE) - CAST(avg(ds_lon)  AS DOUBLE))
) as dist
from a
group by 1
)

select * from city_level
-- [DEBUG POINT 7]: Distance Threshold
-- Sample Failure: Make sure 100km is the correct business logic threshold for a satellite city.
where dist>100
),

---------------------------------------------------------------------------------------------------------------------------------------
sale_base as (
select pi.id,pi.insert_ds_ist,pi.actual_sales,pi.created_at,pi.grofers_order_id,pi.invoice_id,
case when invoice_type_id in (1,19) then 'Sales' else 'Return' end as type_,
pi.outlet_id,
pi.outlet_name,
outlet_city
from pos.pos_invoice pi
join supply_etls.outlet_details od on od.hot_outlet_id=pi.outlet_id
-- [DEBUG POINT 8]: City String Join (VERY HIGH RISK OF FAILURE)
-- Sample Failure: pi.outlet_city = 'Delhi' but sc.city_name = 'New Delhi' or 'Delhi '.
-- Fix: join sat_city sc on LOWER(TRIM(pi.outlet_city)) = LOWER(TRIM(sc.city_name))
join sat_city sc on  pi.outlet_city=sc.city_name
-- [DEBUG POINT 9]: Date Casting
-- Sample Failure: If insert_ds_ist is a DATE type and python passes '2026-04-01' as a VARCHAR, 
-- Trino might fail the query. 
-- Fix: where insert_ds_ist between DATE '{{start_date}}' and DATE '{{end_date}}'
where insert_ds_ist  between '{{start_date}}' and '{{end_date}}'
and invoice_type_id in (1,2,19,20)
and lake_active_record
),

p_type as (
select *,
case when storage_type in ('Non_Veg_Cold','Freezer','Fridge','Non_Veg_Frozen') 
or handling_type_id = 5
then 'Cold' else 'Other Products' end as p_type_calc
from supply_etls.item_details
),

item_sr as (
select a.*,
pip.upc_id,
pip.variant_id,
pip.name as item_name,
pip.quantity,
od.facility_id,od.facility_name,
id.manufacturer_id as manufacturer_id_x,
id.manufacturer_name as manufacturer_name_x,
rp.variant_mrp,
sf.*,
id.p_type_calc
from sale_base a
join pos.pos_invoice_product_details pip on pip.invoice_id=a.id
join rpc.product_product rp ON rp.variant_id = pip.variant_id
join p_type id on id.item_id=rp.item_id
-- [DEBUG POINT 10]: Product Type Join (HIGH RISK OF FAILURE)
-- Sample Failure: id.p_type_calc yields 'Cold'. If sf.product_type from JSON yields 'cold' (lowercase), 
-- this join completely wipes out all sales data. 
-- Fix: and LOWER(TRIM(id.p_type_calc)) = LOWER(TRIM(sf.product_type))
join sf on sf.manufacturer_id=cast(id.manufacturer_id as varchar) and id.p_type_calc = sf.product_type
join supply_etls.outlet_details od on od.hot_outlet_id=a.outlet_id
where pip.insert_ds_ist between '{{start_date}}' and '{{end_date}}'
and pip.lake_active_record
),

item_sat_fee as (
select 
insert_ds_ist,
manufacturer_id_x,
manufacturer_name_x,
outlet_id,
outlet_name,
outlet_city,
facility_id,
facility_name,
variant_id,
item_name,
variant_mrp,
item_mrp as mrp_threshold,
contract_state,
product_type,
bucket,
fee_amt,
coalesce(sum(quantity*1) filter(where type_='Sales'),0)  as qty_sold,
coalesce(sum(quantity*1) filter(where type_='Return'),0) as qty_returned,
coalesce(sum(quantity*1) filter(where type_='Sales'),0) - coalesce(sum(quantity*1) filter(where type_='Return'),0) as net_qty_sold,

-- [DEBUG POINT 11]: Logic Error in MRP Bucket Math (CRITICAL BUG)
-- Existing code says: WHEN bucket = 'Item MRP < X' AND item_mrp*1 < variant_mrp*1
-- This means if the threshold (item_mrp) is 200, and actual price (variant_mrp) is 50, 
-- it evaluates 200 < 50 (False). It falls to ELSE 0, and gets filtered out entirely below.
-- The fix below corrects the greater-than/less-than signs:
case 
    -- Fix: When actual price is LESS THAN the threshold
    when bucket='Item MRP < X' and (variant_mrp*1) < (item_mrp*1) then 
        (coalesce(sum(quantity*1) filter(where type_='Sales'),0) - coalesce(sum(quantity*1) filter(where type_='Return'),0))*(fee_amt*1)
    
    -- Fix: When actual price is GREATER THAN OR EQUAL TO the threshold
    when bucket<>'Item MRP < X' and (variant_mrp*1) >= (item_mrp*1) then 
        (coalesce(sum(quantity*1) filter(where type_='Sales'),0) - coalesce(sum(quantity*1) filter(where type_='Return'),0))*(fee_amt*1)
else 0 end as sat_fee_applicable

from item_sr
group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
)

select *
from item_sat_fee
-- [DEBUG POINT 12]: The Final Filter
-- Sample Failure: If any of the bugs above trigger (casing mismatch, bad join, backwards math), 
-- sat_fee_applicable will be 0 or NULL. This line deletes all those rows, resulting in an empty file.
where sat_fee_applicable>0


"""

print(MY_QUERY)

# Name your output file (it will be saved in the OUTPUT_DIR defined in Cell 1)
OUTPUT_FILENAME = "my_custom_extract.parquet"

# Run the function
execute_and_export(query=MY_QUERY, filename=OUTPUT_FILENAME)